our friend Eliud inherited a farm from her grandma Tigist. Her granny was an inventor and had a tendency to build things in an overly complicated manner. The chicken coop has a digital display showing an encoded number representing the positions of all eggs that could be picked up.

Eliud is asking you to write a program that shows the actual number of eggs in the coop.

The position information encoding is calculated as follows:

Scan the potential egg-laying spots and mark down a 1 for an existing egg or a 0 for an empty spot.
Convert the number from binary to decimal.
Show the result on the display.
Example 1
Seven individual nest boxes arranged in a row whose first, third, fourth and seventh nests each have a single egg.

 _ _ _ _ _ _ _
|E| |E|E| | |E|
Resulting Binary
1011001

 _ _ _ _ _ _ _
|1|0|1|1|0|0|1|
Decimal number on the display
89

Actual eggs in the coop
4

Example 2
Seven individual nest boxes arranged in a row where only the fourth nest has an egg.

 _ _ _ _ _ _ _
| | | |E| | | |
Resulting Binary
0001000

 _ _ _ _ _ _ _
|0|0|0|1|0|0|0|
Decimal number on the display
8

Actual eggs in the coop
1

Instructions
Your task is to count the number of 1 bits in the binary representation of a number.

Restrictions
Keep your hands off that bit-count functionality provided by your standard library! Solve this one yourself using other basic tools instead.

## Approach

The display number encodes egg positions as binary digits: a `1` is an egg, a `0` is an
empty nest. So "how many eggs?" is really **"how many 1-bits are in this number?"** — the
decimal value itself doesn't matter, only the count of set bits.

I explored two iterations of the same idea (reading the binary digits without using a
built-in bit-count):

- **Iteration 1 — top-down:** repeatedly subtract the largest power of two that fits, counting
  one bit per subtraction.
- **Iteration 2 — bottom-up:** repeatedly take `n % 2` to read the lowest bit and `n // 2` to
  shift to the next, counting each `1` remainder.

The key fact behind both: a number is a sum of powers of two
(`89 = 64 + 16 + 8 + 1`), and each power of two present contributes exactly one `1` bit.

### Iteration 1 — subtract the largest power of two

By hand, I would step through `2, 4, 8, 16, …` until the value exceeds the input, place a
binary digit at that position, subtract it from the total, and repeat on the remainder until
the whole number is accounted for. Each subtraction corresponds to one `1` bit, so I just
count the subtractions.

**Bug I found and fixed:** I originally started `binary_value` at `2` and reset it to `2`
each pass. That made the units bit (value `1`) invisible — `remaining` could never reach
exactly `0`, so the loop ran off into negative numbers. I had patched it with three special
cases (`== 0`, `== 1`, and an in-loop `remaining == 1`). Starting `binary_value` at `1`
instead fixed the root cause and let me delete all three patches.

**Cost:** the inner `while` loop re-discovers the largest power of two each pass by doubling
up from `1`, so this does a search inside a search — roughly `O(k · log n)` for a number with
`k` set bits.

In [67]:
def eliuds(display_v):

    remaining_decimal = display_v
    binary_sum = 0
    number_of_bits = 0
    binary_value = 1

    while binary_sum != display_v:  

        while binary_value*2 <= remaining_decimal:
            binary_value = binary_value*2

        binary_sum += binary_value
        number_of_bits += 1
        remaining_decimal = remaining_decimal - binary_value
        binary_value = 1



    return number_of_bits


    

In [65]:
eliuds(3)

2

### Iteration 2 — divide by two and read remainders

Instead of hunting for the biggest power of two from the top, work from the bottom:

- `n % 2` reads the lowest bit directly — it's `1` when `n` is odd, `0` when even.
- `n // 2` shifts every bit one place right, dropping the bit just read and sliding the next
  one into the units position (the base-2 version of `1234 // 10 == 123`).

Repeating until `n == 0` visits every bit from least to most significant. Counting eggs means
just adding up the remainders (each `0` or `1`) — no need to rebuild the number or store the
binary string.

For `89` the remainders fall out as `1, 0, 0, 1, 1, 0, 1` (the binary `1011001` read
backwards), and their sum is `4`.

**Cost:** one division per bit and no inner search loop, so `O(log n)` — simpler and a touch
cheaper than iteration 1.

In [68]:
def eliuds(display_v):

    remaining_decimal = display_v
    number_of_bits = 0
    

    while remaining_decimal != 0:

        if remaining_decimal%2 == 1:
            number_of_bits += 1
        remaining_decimal = remaining_decimal//2

    return number_of_bits


In [71]:
eliuds(89)

4